# 01 · Hello, Gradient Descent!
### *No biology required — just a function and a slope*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins-lab/diff-biophys/blob/main/examples/interactive_tutorials/01_hello_gradient_descent.ipynb)

---

**Who is this for?**  Anyone who has taken calculus and is curious about how machine learning "learns."
You do *not* need a biology background for this notebook.

**What you will learn:**
1. What a gradient is (visually and mathematically)
2. How gradient descent minimises a function — step by step
3. How JAX computes gradients *automatically* (no more hand-derivations!)
4. How to fit a real scientific equation to data using the exact same idea

**Time:** ~30 minutes


## 0 · Setup

Install the library (one-time, fast on Colab).

In [ ]:
# Install diff-biophys and plotting tools
%pip install -q diff-biophys==0.1.5 matplotlib

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Pretty plots
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 12,
})
print("JAX version:", jax.__version__)
print("Devices available:", jax.devices())

---
## 1 · What is a gradient?

A **gradient** is the slope of a function at a point.

For a one-variable function $f(x)$, the gradient is just the derivative $\frac{df}{dx}$.
It tells you: *"if I increase $x$ a little, does $f$ go up or down, and by how much?"*

### A simple example

Let's use the **parabola** $f(x) = (x - 3)^2$.
- It has a minimum at $x = 3$ (where $f = 0$).
- Its gradient is $\frac{df}{dx} = 2(x - 3)$.
  - At $x = 0$: gradient = $-6$ (slope is negative, $f$ decreases as $x$ increases)
  - At $x = 5$: gradient = $+4$ (slope is positive, $f$ increases as $x$ increases)
  - At $x = 3$: gradient = $0$  ← minimum!


In [ ]:
x_vals = np.linspace(-1, 7, 300)
f_vals = (x_vals - 3) ** 2

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_vals, f_vals, "steelblue", lw=2.5, label=r"$f(x) = (x-3)^2$")
ax.axvline(3, color="crimson", lw=1.5, ls="--", label="minimum at x = 3")

# Draw gradient arrows at a few points
for x0 in [0.5, 2.0, 4.5, 6.0]:
    g = 2 * (x0 - 3)          # analytic gradient
    f0 = (x0 - 3) ** 2
    ax.annotate("", xy=(x0 + 0.6 * np.sign(g), f0 + 0.6 * abs(g)),
                xytext=(x0, f0),
                arrowprops=dict(arrowstyle="->", color="darkorange", lw=2))

ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("The parabola and its gradients")
ax.legend()
plt.tight_layout()
plt.show()
print("Gradient at x=0:", 2 * (0 - 3))
print("Gradient at x=5:", 2 * (5 - 3))
print("Gradient at x=3:", 2 * (3 - 3))

---
## 2 · Gradient descent — the algorithm

Here is the entire idea in one sentence:

> **Take a small step in the direction that makes f decrease.**

Since the gradient points *uphill*, we step in the *opposite* direction:

$$x_{t+1} = x_t - \eta \cdot \frac{df}{dx}\bigg|_{x_t}$$

$\eta$ (eta) is the **learning rate** — how big each step is.

Let's watch it work, starting at $x_0 = 0$:


In [ ]:
def f(x):
    return (x - 3) ** 2

def grad_f(x):
    return 2 * (x - 3)          # hand-computed derivative

learning_rate = 0.15
x = 0.0                         # starting point
trajectory = [x]

for step in range(25):
    g = grad_f(x)
    x = x - learning_rate * g   # gradient descent update
    trajectory.append(x)

# Plot the trajectory on the parabola
x_vals = np.linspace(-0.5, 6.5, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(x_vals, f(x_vals), "steelblue", lw=2.5)
ax1.scatter(trajectory, [f(x) for x in trajectory],
            c=range(len(trajectory)), cmap="plasma", s=50, zorder=5)
ax1.axvline(3, color="crimson", lw=1.5, ls="--", alpha=0.5)
ax1.set_xlabel("x"); ax1.set_ylabel("f(x)")
ax1.set_title("Gradient descent path on the parabola")

ax2.plot(range(len(trajectory)), [f(x) for x in trajectory],
         "steelblue", lw=2, marker="o", ms=4)
ax2.set_xlabel("Step"); ax2.set_ylabel("f(x)  [loss]")
ax2.set_title("Loss vs. step")
plt.tight_layout()
plt.show()

print(f"Started at x = 0.00,  f = {f(0.0):.2f}")
print(f"Ended   at x = {trajectory[-1]:.4f},  f = {f(trajectory[-1]):.6f}")

---
## 3 · JAX does the calculus for you

Hand-computing the derivative works for $(x-3)^2$.
But what about a complicated function with hundreds of inputs?

That's where **automatic differentiation** (autodiff) comes in.
JAX can compute the exact derivative of *any* differentiable function,
automatically, in one line:

```python
grad_f = jax.grad(f)
```

No chain rule by hand. No finite differences. **Exact gradients, always.**


In [ ]:
# Define the function with JAX arrays
def f_jax(x):
    return (x - 3.0) ** 2

# jax.grad returns a NEW function that computes the gradient
grad_f_jax = jax.grad(f_jax)

# Test it at a few points
for x0 in [0.0, 3.0, 5.0]:
    x0_jax = jnp.array(x0)
    g_auto  = grad_f_jax(x0_jax)
    g_hand  = 2 * (x0 - 3)
    print(f"x={x0:4.1f}  →  auto-grad={float(g_auto):+.2f}  hand-calc={g_hand:+.2f}  ✓ match: {abs(float(g_auto) - g_hand) < 1e-5}")

In [ ]:
# Works on multi-dimensional functions too
def f_2d(params):
    """f(x, y) = (x - 3)^2 + (y + 1)^2   minimum at (3, -1)"""
    x, y = params[0], params[1]
    return (x - 3.0) ** 2 + (y + 1.0) ** 2

grad_f_2d = jax.grad(f_2d)

params = jnp.array([0.0, 0.0])
for step in range(30):
    g = grad_f_2d(params)
    params = params - 0.2 * g

print(f"Minimum found at:  x = {params[0]:.4f}, y = {params[1]:.4f}")
print(f"True minimum is:   x = 3.0000, y = -1.0000")

In [ ]:
# Also JIT-compilable for speed — same code, 10-100x faster on GPU
@jax.jit
def step(params, lr=0.2):
    return params - lr * jax.grad(f_2d)(params)

params = jnp.array([0.0, 0.0])
for _ in range(30):
    params = step(params)
print(f"JIT result:  x = {params[0]:.4f}, y = {params[1]:.4f}")

---
## 4 · Fitting a real scientific equation

Here is where it gets exciting for biology.

**The Karplus equation** describes how a measurable NMR coupling constant $J$ (in Hz)
depends on a bond angle $\theta$:

$$J(\theta) = A \cos^2(\theta) + B \cos(\theta) + C$$

Scientists measure $J$ values experimentally and want to recover
the coefficients $A$, $B$, $C$ — this is a **curve-fitting** problem.

Gradient descent solves this automatically.
We minimize the **mean-squared error** between our predicted $J$ values and the measured ones.


In [ ]:
from diff_biophys.nmr.karplus import calculate_karplus_j

# --- True coefficients (Vuister & Bax 1993 for HN-Hα coupling) ---
A_true, B_true, C_true = 6.98, -1.38, 1.72

# --- Generate synthetic "experimental" data ---
rng = np.random.default_rng(42)
theta_data = jnp.array(rng.uniform(0, np.pi, 30), dtype=jnp.float32)
J_exp = calculate_karplus_j(theta_data, A_true, B_true, C_true)
J_exp = J_exp + jnp.array(rng.normal(0, 0.15, 30), dtype=jnp.float32)  # add noise

# Plot the data
theta_dense = jnp.linspace(0, jnp.pi, 300)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.degrees(theta_dense),
        calculate_karplus_j(theta_dense, A_true, B_true, C_true),
        "steelblue", lw=2, label="True Karplus curve")
ax.scatter(np.degrees(theta_data), J_exp, color="crimson", s=40,
           zorder=5, label="Noisy observations")
ax.set_xlabel("θ (degrees)"); ax.set_ylabel("J (Hz)")
ax.set_title("Synthetic Karplus data — can we recover A, B, C?")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
import optax   # pip install optax  (already a dep in diff-biophys[dev])

# --- Define the loss function ---
def karplus_loss(params):
    """MSE between predicted and experimental J-couplings."""
    A, B, C = params[0], params[1], params[2]
    J_pred = calculate_karplus_j(theta_data, A, B, C)
    return jnp.mean((J_pred - J_exp) ** 2)

# --- Initialise with wrong parameters ---
params = jnp.array([3.0, 0.0, 0.5], dtype=jnp.float32)
print(f"Starting loss:  {karplus_loss(params):.4f} Hz²")
print(f"Starting A={params[0]:.2f}, B={params[1]:.2f}, C={params[2]:.2f}")
print(f"True    A={A_true:.2f}, B={B_true:.2f}, C={C_true:.2f}\n")

# --- Adam optimiser ---
optimizer = optax.adam(learning_rate=0.05)
opt_state = optimizer.init(params)
grad_loss  = jax.jit(jax.value_and_grad(karplus_loss))

losses = []
for step in range(300):
    loss_val, grads = grad_loss(params)
    losses.append(float(loss_val))
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

print(f"Final   loss:  {losses[-1]:.4f} Hz²")
print(f"Recovered A={params[0]:.2f}, B={params[1]:.2f}, C={params[2]:.2f}")
print(f"True      A={A_true:.2f}, B={B_true:.2f}, C={C_true:.2f}")

In [ ]:
# Plot: loss curve + recovered fit
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogy(losses, "steelblue", lw=2)
ax1.set_xlabel("Optimisation step"); ax1.set_ylabel("MSE loss (Hz²)")
ax1.set_title("Loss decreasing during gradient descent")

A_fit, B_fit, C_fit = float(params[0]), float(params[1]), float(params[2])
ax2.plot(np.degrees(theta_dense),
         calculate_karplus_j(theta_dense, A_true, B_true, C_true),
         "steelblue", lw=2, label="True curve")
ax2.plot(np.degrees(theta_dense),
         calculate_karplus_j(theta_dense, A_fit, B_fit, C_fit),
         "crimson", lw=2, ls="--", label="Fitted curve")
ax2.scatter(np.degrees(theta_data), J_exp, color="grey", s=30, alpha=0.6)
ax2.set_xlabel("θ (degrees)"); ax2.set_ylabel("J (Hz)")
ax2.set_title("True vs. recovered Karplus curve")
ax2.legend(); plt.tight_layout(); plt.show()
print("\n✅ Gradient descent recovered the Karplus coefficients from noisy data!")

---
## 5 · Summary

| Concept | One-line version |
|---|---|
| **Gradient** | The slope — which direction is uphill, and how steep? |
| **Gradient descent** | Take small steps downhill until you reach the bottom |
| **`jax.grad`** | Computes exact gradients of any function automatically |
| **`jax.jit`** | Compiles the function for GPU speed |
| **Karplus fitting** | Real science, same idea — minimize MSE over parameters |

### What's next?
- **Notebook 02 — NMR Fundamentals**: apply these ideas to chemical shifts and RDCs
- **Notebook 03 — CD Spectroscopy**: differentiate a coupled-oscillator model of helical dichroism

> 💡 **Key insight**: `diff-biophys` gives you the *same* gradient machinery you just used,
> but wired up to full protein structures with hundreds of atoms.
> The physics is more complex; the idea is identical.
